In [27]:
# ============================================================
# INGEST SPRINTS — LOCAL EXECUTION (FINAL FIXED VERSION)
# ============================================================

import os
import json
import nbformat
from nbconvert import PythonExporter
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, DateType
)

In [28]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG
# -----------------------------------------
try:
    BRONZE_LANDING
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

In [29]:
# -----------------------------------------
# 2. IMPORT HELPER FUNCTIONS
# -----------------------------------------
try:
    add_ingestion_metadata
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    helper_path = "/Users/manoharazalki/F1-Analytics/notebooks/00-common/02_bronze_helpers.ipynb"
    with open(helper_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

In [30]:
# -----------------------------------------
# 3. Define source + target paths
# -----------------------------------------
source_folder = f"{BRONZE_LANDING}/sprints"
target_folder = f"{BRONZE_ROOT}/sprints"
target_file = f"{target_folder}/sprints.parquet"

os.makedirs(target_folder, exist_ok=True)

print("Source folder:", source_folder)
print("Target file:", target_file)

Source folder: /Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/sprints
Target file: /Users/manoharazalki/F1-ANALYTICS/data/bronze/sprints/sprints.parquet


In [31]:
# -----------------------------------------
# 4. Convert JSON arrays → JSON lines
# -----------------------------------------
clean_files = []

for file in os.listdir(source_folder):
    if not file.endswith(".json"):
        continue

    input_path = f"{source_folder}/{file}"
    output_path = f"{source_folder}/clean_{file}"

    try:
        with open(input_path, "r") as f:
            data = json.load(f)

        # If file is a JSON array → convert to JSON lines
        if isinstance(data, list):
            with open(output_path, "w") as out:
                for row in data:
                    out.write(json.dumps(row) + "\n")
            clean_files.append(output_path)
            print(f"✔ Converted array JSON → JSON lines: {file}")

        # If file is already JSON lines → keep as is
        else:
            clean_files.append(input_path)
            print(f"✔ JSON lines file detected: {file}")

    except Exception as e:
        print(f"Skipping invalid JSON file: {file}")
        print("Error:", e)

print("Clean files to load:", clean_files)

Skipping invalid JSON file: clean_sprints_2024.json
Error: Extra data: line 2 column 1 (char 317)
✔ Converted array JSON → JSON lines: sprints_2021.json
Skipping invalid JSON file: clean_sprints_2025.json
Error: Extra data: line 2 column 1 (char 311)
Skipping invalid JSON file: clean_sprints_2022.json
Error: Extra data: line 2 column 1 (char 331)
Skipping invalid JSON file: clean_sprints_2023.json
Error: Extra data: line 2 column 1 (char 315)
✔ Converted array JSON → JSON lines: sprints_2024.json
Skipping invalid JSON file: clean_sprints_2021.json
Error: Extra data: line 2 column 1 (char 319)
✔ Converted array JSON → JSON lines: sprints_2025.json
✔ Converted array JSON → JSON lines: sprints_2022.json
✔ Converted array JSON → JSON lines: sprints_2023.json
Clean files to load: ['/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/sprints/clean_sprints_2021.json', '/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/sprints/clean_sprints_2024.json', '/Users/manoharazalki/F1-ANALYTICS/

In [32]:
# -----------------------------------------
# 5. Validate JSON files (catch malformed ones)
# -----------------------------------------
valid_files = []
invalid_files = []

json_files = [f for f in os.listdir(source_folder) if f.endswith(".json")]
for file in json_files:
    path = f"{source_folder}/{file}"
    try:
        with open(path, "r") as f:
            json.load(f)
        valid_files.append(path)
    except Exception as e:
        print(f"Invalid JSON file skipped: {file}")
        print("   Error:", e)
        invalid_files.append(file)

print("Valid JSON files:", valid_files)
print("Invalid JSON files:", invalid_files)

if not valid_files:
    raise Exception("No valid JSON files to load!")

Invalid JSON file skipped: clean_sprints_2024.json
   Error: Extra data: line 2 column 1 (char 317)
Invalid JSON file skipped: clean_sprints_2025.json
   Error: Extra data: line 2 column 1 (char 311)
Invalid JSON file skipped: clean_sprints_2022.json
   Error: Extra data: line 2 column 1 (char 331)
Invalid JSON file skipped: clean_sprints_2023.json
   Error: Extra data: line 2 column 1 (char 315)
Invalid JSON file skipped: clean_sprints_2021.json
   Error: Extra data: line 2 column 1 (char 319)
Valid JSON files: ['/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/sprints/sprints_2021.json', '/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/sprints/sprints_2024.json', '/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/sprints/sprints_2025.json', '/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/sprints/sprints_2022.json', '/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/sprints/sprints_2023.json']
Invalid JSON files: ['clean_sprints_2024.json', 'clean_sprints_202

In [33]:
# -----------------------------------------
# 5. Define schema
# -----------------------------------------
sprint_schema = StructType([
    StructField("date", DateType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", FloatType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True)
])

In [34]:
# -----------------------------------------
# 6. Read cleaned JSON files
# -----------------------------------------
sprints_df = (
    spark.read
        .format("json")
        .schema(sprint_schema)
        .option("mode", "FAILFAST")
        .load(clean_files)
)

print("✔ Sprints JSON loaded successfully")
sprints_df.show(5, truncate=False)

✔ Sprints JSON loaded successfully
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+
|date      |raceName             |round|season|url                                                     |constructorId|driverId      |grid|laps|number|points|position|positionText|status  |
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+
|2023-04-30|azerbaijan grand prix|4    |2023  |https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix|red_bull     |perez         |2   |17  |11    |8.0   |1       |1           |Finished|
|2023-04-30|azerbaijan grand prix|4    |2023  |https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix|ferrari      |leclerc       |1   |17  |16    |7.0   |2       |2           |Finished|
|2023-04-30|azerbaij

In [35]:
# -----------------------------------------
# 7. Add metadata
# -----------------------------------------
sprints_final_df = add_ingestion_metadata(sprints_df, source_folder)

print("✔ Metadata added")
sprints_final_df.show(5, truncate=False)

✔ Metadata added
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+--------------------------+-------------------------------------------------------------+
|date      |raceName             |round|season|url                                                     |constructorId|driverId      |grid|laps|number|points|position|positionText|status  |ingest_timestamp          |source_file                                                  |
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+--------------------------+-------------------------------------------------------------+
|2023-04-30|azerbaijan grand prix|4    |2023  |https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix|red_bull     |perez         |2   |17  |11    

In [36]:
# -----------------------------------------
# 8. Write to Bronze (Option C)
# -----------------------------------------
(
    sprints_final_df
        .write
        .mode("overwrite")
        .parquet(target_file)
)

print(f"✔ Sprints Bronze written to: {target_file}")

✔ Sprints Bronze written to: /Users/manoharazalki/F1-ANALYTICS/data/bronze/sprints/sprints.parquet


In [37]:
# -----------------------------------------
# 9. Read back for validation
# -----------------------------------------
spark.read.parquet(target_file).show(5, truncate=False)

+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+-------------------------+-------------------------------------------------------------+
|date      |raceName             |round|season|url                                                     |constructorId|driverId      |grid|laps|number|points|position|positionText|status  |ingest_timestamp         |source_file                                                  |
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+-------------------------+-------------------------------------------------------------+
|2023-04-30|azerbaijan grand prix|4    |2023  |https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix|red_bull     |perez         |2   |17  |11    |8.0   |1       |1  